<a href="https://colab.research.google.com/github/kaushikpatriot/ML-Projects/blob/main/Bagging_and_Boosting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

###Bagging - Random Forest

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from collections import Counter
from sklearn.tree import DecisionTreeClassifier

In [ ]:
def bag(X,y):
  n_samples = X.shape[0]
  idx = np.random.choice(n_samples,size=n_samples,replace=True)
  return X[idx], y[idx]

def most_common_label(y):
  counter = Counter(y)
  most_common = counter.most_common(1)[0][0]
  return most_common

class RandomForest:
  def __init__(self, n_trees=10,min_samples_split = 2, max_depth = 100, max_features = None):
    self.n_trees = n_trees
    self.min_samples_split = min_samples_split
    self.max_depth = max_depth
    self.max_features = max_features
    self.trees = []

  def fit(self, X,y):
    self.trees = []
    for _ in range(self.n_trees):
      tree = DecisionTreeClassifier(
              min_samples_split = self.min_samples_split,
              max_depth = self.max_depth,
              max_features = self.max_features,
              random_state = 1)
      X_sample, y_sample = bag(X,y)
      tree.fit(X_sample,y_sample)
      self.trees.append(tree)

  def predict(self, X):
    tree_predict = np.array([tree.predict(X) for tree in self.trees])
    tree_predict = np.swapaxes(tree_predict,0,1)
    y_pred = [most_common_label(tree_pred) for tree_pred in tree_predict]
    return np.array(y_pred)


In [ ]:
X = np.random.normal(0,1,100).reshape(-1,2)
y = np.random.choice([0,1],size = 50, replace=True)
X.shape,y.shape, y[:10]
X_bag, y_bag = bag(X,y)
X_bag.shape, y_bag.shape

((50, 2), (50,))

In [ ]:
most_common_label(y)

1

###Boosting - GradBoost

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeRegressor

In [ ]:
def GradBoost(model, X_train, y_train, X_test, boosting_rounds, learning_rate: float = 0.1):
  y_hat_train = np.repeat(np.mean(y_train),len(y_train))
  y_hat_test = np.repeat(np.mean(y_train),len(X_test))
  residuals = y_train - y_hat_train

  for i in range(boosting_rounds):
    model = model.fit(X_train, residuals)

    y_hat_train = y_hat_train + learning_rate * model.predict(X_train)
    y_hat_test = y_hat_test + learning_rate * model.predict(X_test)
    residuals = y_train - y_hat_train

  return y_hat_train, y_hat_test



In [ ]:
from sklearn.datasets import make_regression
X,y = make_regression(n_samples = 1000,
                      n_features = 20,
                      n_informative = 15,
                      n_targets = 1,
                      bias = 0.0,
                      noise = 20,
                      shuffle = True,
                      random_state = 13)

X_train, y_train = X[:800], y[:800]
X_test, y_test = X[800:],y[800:]


In [ ]:
model = DecisionTreeRegressor(criterion = 'squared_error', max_depth = 3)